# 📅 2026-04-05 (일요일) 개발 노트 : gpt-5.4 분석 엔진 구축 및 Batch API 보안 이륙

## 🎯 오늘의 목표
- [x] `.env` 환경변수 보안 체계 구축 (GitHub 유출 방지 및 터미널 직접 노출 차단)
- [x] 4,190개 게임 데이터를 250개씩 분할하여 OpenAI Batch API 전송 성공
- [x] gpt-5.4 모델 기반 52개 정밀 지표 데이터(JSONL) 정상 수신 확인

## 🛠 진행 상황 및 핵심 기록
- **Batch 전송 엔진 (`auto_batch_sender.py`) 구축**: `batch_processor`의 논리적 오류(입력값을 결과값으로 오인)를 해결하고, OpenAI SDK를 직접 호출하여 1시간 간격 자동 업로드 로직 완성.
- **데이터 구조**: `metrics`(52개 지표), `tags`, `content`(마케팅 문구), `reasoning`(분석 근거)을 포함한 고밀도 JSON 구조 확보.
- **인프라**: Docker를 활용하여 `hidden_gem_vectordb`(PostgreSQL) 가동, 데이터 적재 준비 완료.
- **비용 효율**: 250개당 **3.73달러** 소모 확인. 전체 4,190개 분석 시 약 9만원대로 예산(10만원) 내 집행 가능 확인.

## 🚨 트러블슈팅 (문제 및 해결)
- **문제**: `auto_batch_sender.py` 실행 시 "성공 0, 실패 250" 메시지 출력 및 대시보드 미표시.
- **원인**: 기존 코드가 '업로드'용이 아닌 '결과 처리'용인 `batch_processor`를 호출함. 질문지(Input)를 답안지(Output)로 인식해 파싱 에러 발생.
- **해결**: `batch_processor` 호출부를 삭제하고, `client.batches.create`를 직접 사용하는 순수 업로드 스크립트로 전면 수정하여 해결.
- **에러**: `bash: !': event not found` (리눅스 터미널 특수문자 에러). 큰따옴표 내 `!` 제거 후 실행하여 API 키 로드 여부 검증 성공.

## 💡 인사이트 및 다음 할 일
- **인사이트**:
    - **Data Flywheel**: 4,190개의 gpt-5.4 정밀 데이터를 '표준 잣대'로 삼으면, 향후 수십만 개의 게임 리뷰를 NLP로 분석할 때 강력한 기준점이 됨.
    - **Batch API**: `completion_window="24h"` 설정만으로 50% 비용 절감이 가능하며, 대량 데이터 처리에 압도적으로 유리함.
- **다음 할 일**:
    - 17개 파트 완료 확인 후 결과 파일(`.jsonl`) 다운로드 및 병합.
    - `batch_processor.py`를 가동하여 도커 DB에 최종 데이터 적재.
    - 52개 지표를 활용한 **'유저 페르소나 매칭 가중치'** 및 **'입체적 레이더 차트'** 시각화 로직 설계.

In [ ]:
# 핵심 코드 블록
import os
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# [보안] 프로젝트 루트의 .env에서 API_KEY를 안전하게 로드
PROJECT_ROOT = Path(__file__).resolve().parent.parent
load_dotenv(PROJECT_ROOT / ".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def run_real_upload(file_path):
    print(f"🚀 [진짜 업로드 시작] {file_path.name}")
    try:
        # A. 파일 업로드 (Purpose: batch)
        with open(file_path, "rb") as f:
            uploaded_file = client.files.create(file=f, purpose="batch")
        
        # B. 배치 작업 생성 (24h 윈도우 적용으로 50% 할인)
        batch_job = client.batches.create(
            input_file_id=uploaded_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h"
        )
        print(f"   🎊 배치 생성 성공! ID: {batch_job.id} | 상태: {batch_job.status}")
        return True
    except Exception as e:
        print(f"   ❌ 업로드 실패: {str(e)}")
        return False

# 실행 로직: data 폴더 내의 모든 파트 파일을 1시간 간격으로 전송
DATA_DIR = PROJECT_ROOT / "data"
PART_FILES = sorted(list(DATA_DIR.glob("*_part*.jsonl")))

for i, path in enumerate(PART_FILES):
    if run_real_upload(path):
        if i < len(PART_FILES) - 1:
            print("💤 다음 파트까지 1시간 대기합니다...")
            time.sleep(3600)

## 인프라 가동

In [ ]:
# 터미널에서 실행
docker-compose up -d

# 결과: ✔ Container hidden_gem_vectordb Running